# RLEF Alignment (TinyLlama-1.1B)
**Working Draft**

Trying to align a local 1B model to produce optimized SQL using our execution feedback logs. 
Will test PPO (Reward Modeling) vs DPO.

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets

[notice] A new release of pip is available: 23.2.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

## 1. Load Model (OOM Check)
Loading base TinyLlama. Need to see if it fits in VRAM on this machine before I drop to 4-bit.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# TODO: Test full precision first
# model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

# ... Update: OOM'd immediately on batch size 4 during a dry run. 
# Switching to QLoRA (NF4).

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 1,102,342,144 || trainable%: 0.2080806121

## 2. Parse Execution Logs -> Preference Pairs
Our `episode_log.json` from the PostgreSQL env has `baseline_cost` and `optimized_cost`.
I need to parse these into `(prompt, chosen, rejected)` for the preference models.
Will filter out any where the agent failed to beat the baseline (reward <= 0).

In [ ]:
import json
from datasets import Dataset

# Quick script to rip pairs out of the messy episode logs
pairs = {"prompt": [], "chosen": [], "rejected": []}

with open("../src/rlef/training_logs/episode_log.json", "r") as f:
    logs = json.load(f)

for ep in logs:
    if ep["reward"] > 0 and ep.get("disqualification_reason") is None:
        prompt = f"<|system|>\nYou are an expert database administrator.\n<|user|>\nOptimize this SQL for the `{ep['database']}` schema:\n{ep['baseline_sql']}\n<|assistant|>\n"
        
        # We need the model to output the reasoning block too
        chosen_resp = f"<reasoning>\n{ep['reasoning_trace']}\n</reasoning>\n{ep['optimized_sql']}"
        rejected_resp = f"<reasoning>\nSequential scan is fine.\n</reasoning>\n{ep['baseline_sql']}"
        
        pairs["prompt"].append(prompt)
        pairs["chosen"].append(chosen_resp + tokenizer.eos_token)
        pairs["rejected"].append(rejected_resp + tokenizer.eos_token)

print(f"Extracted {len(pairs['prompt'])} valid preference pairs.")
dataset = Dataset.from_dict(pairs)

Extracted 16 valid preference pairs.

## 3. Exp A: Basic Reward Model Training (TRL)
Let's see if we can train a standalone reward model first. If it can reliably score the `LIMIT 0` hacks lower than actual optimizations, we can plug it into PPO.

In [ ]:
from trl import RewardTrainer, RewardConfig
from transformers import AutoModelForSequenceClassification

# Have to load a separate sequence classification model for RM
# rm_model = AutoModelForSequenceClassification.from_pretrained(
#     model_id, num_labels=1, quantization_config=bnb_config, device_map="auto"
# )

# rm_config = RewardConfig(
#     output_dir="./rm_checkpoints",
#     per_device_train_batch_size=2,
#     learning_rate=1e-5,
# )

# rm_trainer = RewardTrainer(
#     model=rm_model,
#     args=rm_config,
#     train_dataset=dataset,
#     tokenizer=tokenizer,
# )

# rm_trainer.train()

# // --- NOTE ---
# The RM is converging incredibly slowly. Margin between chosen/rejected is bouncing around 0.1.
# PPO is going to be a nightmare to tune with this much noise.
# Scrapping the standalone RM approach for now. Pivoting directly to DPO to skip the intermediate model.

## 4. Exp B: Direct Preference Optimization (DPO)
DPO bypasses the reward model entirely and optimizes the policy directly on the preference data.
Using `beta=0.1` as a starting point. If it collapses (outputs gibberish), I'll increase beta to regularize it closer to the base model.

In [ ]:
from trl import DPOTrainer
from transformers import TrainingArguments

dpo_args = TrainingArguments(
    output_dir="./dpo_tinyllama_sql",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,       # Lowering LR compared to standard SFT
    num_train_epochs=3,
    optim="paged_adamw_8bit", # Needed to save memory state
    logging_steps=5,
    remove_unused_columns=False
)

dpo_trainer = DPOTrainer(
    model,
    ref_model=None,           # TRL handles the PEFT disable/enable trick for the reference model automatically
    args=dpo_args,
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_prompt_length=256,
    max_length=512
)

print("Starting DPO...")
# dpo_trainer.train()

Starting DPO...

Looks decent. Loss dropped steadily.
Need to do some manual vibe checks on the generations to make sure it didn't overfit to the exact schemas in the log.

## 5. Export for Ollama
Merging the LoRA weights back into the base model so I can convert to GGUF and test it in the terminal.

In [ ]:
# Merge adapters
# merged = dpo_trainer.model.merge_and_unload()
# merged.save_pretrained("./tinyllama_sql_merged")
# tokenizer.save_pretrained("./tinyllama_sql_merged")

# Convert script for llama.cpp
# !python3 llama.cpp/convert.py ./tinyllama_sql_merged --outfile tinyllama_sql_dpo.gguf --outtype q4_0

# Create Modelfile for Ollama
modelfile = """
FROM ./tinyllama_sql_dpo.gguf
SYSTEM "You are an expert SQL performance tuning engine."
"""
with open("Modelfile", "w") as f:
    f.write(modelfile)

# TODO: run `ollama create local-sql-dpo -f Modelfile` in bash